# Getting started

Search biomedical concepts across several knowledge sources with one call, look up a
concept by identifier, and configure the lookup.

The API is asynchronous. Jupyter runs `await` at the top level of a cell; in a script,
wrap the calls in an `async def main()` and run it with `asyncio.run(main())`.

Install the library first:

```bash
pip install biomedical-knowledge-lookup
```

## Create a lookup

`create_knowledge_lookup` builds a `CentralKnowledgeLookup` with the sources you enable.
Public sources need no configuration. A source whose API key or optional extra is missing
is left out of `lookup.adapters` instead of raising an error.

In [ ]:
from knowledge_lookup import (
    CentralKnowledgeLookup,
    ConceptType,
    KnowledgeSource,
    LookupConfig,
    create_knowledge_lookup,
)

lookup = create_knowledge_lookup(
    enabled_sources=[KnowledgeSource.MONDO, KnowledgeSource.HPO, KnowledgeSource.OLS]
)
print("Available:", [source.value for source in lookup.get_available_sources()])

## Search across sources

`search_concepts` queries the sources in parallel, merges concepts that share a label,
and sorts them by confidence. Without `sources=` it uses every available adapter.

In [ ]:
result = await lookup.search_concepts("diabetes", max_results=10)

print(f"{result.total_found} concepts in {result.execution_time:.1f}s")
print("Succeeded:", list(result.sources_succeeded), "Failed:", list(result.sources_failed))
for concept in result.concepts[:5]:
    print(f"- {concept.primary_label} ({concept.primary_id}) [{concept.concept_type}]")

## Narrow a search

Pass `sources=` to query a subset and `concept_types=` to filter by type. Concepts whose
type the adapter could not classify (`UNKNOWN`) are kept by the filter.

In [ ]:
phenotypes = await lookup.search_concepts(
    "seizure",
    sources=[KnowledgeSource.HPO],
    concept_types=[ConceptType.PHENOTYPE],
    max_results=5,
)
for concept in phenotypes.concepts:
    print(f"{concept.primary_id}: {concept.primary_label}")

## Look up a concept by identifier

`get_concept_details(concept_id, source=...)` returns a `UnifiedConcept`, or `None` when
the concept is not found. Identifier formats differ per source: Mondo takes the CURIE
`MONDO:0005148`, OLS takes the full term IRI. The [per-source examples](../README.md)
show a working identifier for each source.

In [ ]:
details = await lookup.get_concept_details("MONDO:0005148", source=KnowledgeSource.MONDO)
if details is None:
    print("Not found")
else:
    print(details.primary_label, details.primary_id)
    print("Definition:", details.definitions[0] if details.definitions else "-")
    print("Synonyms:", ", ".join(details.synonyms[:5]))

## Work with results

`LookupResult` has helpers for picking and grouping concepts, and the lookup can format
a result as a text table.

In [ ]:
for concept in result.get_best_matches(3):
    print(f"{concept.confidence_score:.2f} {concept.primary_label}")

for source, concepts in result.group_by_source().items():
    print(getattr(source, "value", source), len(concepts))

print(lookup.format_results_table(result))
await lookup.close()

## Configure the lookup

`LookupConfig` holds the settings. Pass it to `CentralKnowledgeLookup`, or pass the same
fields as keyword arguments to `create_knowledge_lookup`. Unknown fields are rejected.

In [ ]:
config = LookupConfig(
    enabled_sources=[KnowledgeSource.HPO],
    timeout_per_source=10.0,  # seconds per source
    enable_deduplication=True,  # merge concepts with the same label
)
lookup = CentralKnowledgeLookup(config)
result = await lookup.search_concepts("ataxia", max_results=5)
print([concept.primary_label for concept in result.concepts])
await lookup.close()

## UMLS features

The UMLS adapter adds source-restricted and semantic-type filtered search, bulk search,
mappings, relationships and streaming iterators. These are adapter methods, so call them
on `lookup.adapters[KnowledgeSource.UMLS]`. UMLS needs the `[umls]` extra and the
`UMLS_API_KEY` environment variable; the cell below prints a message when they are missing.

In [ ]:
umls_lookup = create_knowledge_lookup(enabled_sources=[KnowledgeSource.UMLS])
umls = umls_lookup.adapters.get(KnowledgeSource.UMLS)
if umls is None:
    print("UMLS is not available: install the [umls] extra and set UMLS_API_KEY.")
else:
    snomed = await umls.search_concepts("hypertension", limit=3, sabs="SNOMEDCT_US")
    print("SNOMEDCT_US:", [(c.primary_id, c.primary_label) for c in snomed])

    diseases = await umls.search_concepts("diabetes", limit=3, semantic_types="T047")
    print("T047:", [(c.primary_id, c.primary_label) for c in diseases])

    bulk = await umls.bulk_search(["metformin", "atorvastatin"], limit=2)
    print("Bulk:", {query: [c.primary_label for c in hits] for query, hits in bulk.items()})

    mappings = await umls.get_mappings("C0025598", target_source="RXNORM", limit=3)
    print("RXNORM:", [(m["source_id"], m["source_name"]) for m in mappings])

    parents = await umls.get_relationships("C0025598", relation_labels="PAR", limit=3)
    print("Parents:", [relation["related_name"] for relation in parents])

    definitions = [definition async for definition in umls.iter_definitions("C0025598")]
    print("Definitions of metformin:", len(definitions))
await umls_lookup.close()

## Next steps

- [02 API keys](02-api-keys.ipynb): configure the sources that need a key.
- [03 Rate limits, retries and timeouts](03-rate-limiting.ipynb)
- [04 Error handling](04-error-handling.ipynb)
- [Per-source examples](../README.md) and [use cases](../use-cases.md)